# Derived Tables

This notebook follows the M05 class workflow for creating `BOW`, `DTM`, `TFIDF`, and a reduced L2-normalized TFIDF table from the parsed `LIB`, `CORPUS`, and `VOCAB` tables.

# Set Up

## Import

In [1]:
import pandas as pd
import numpy as np

## Config

In [2]:
data_home = "."
output_dir = "output"
data_prefix = "constitutions"
source_dir = output_dir
CSV_DELIM = "|"

In [3]:
OHCO = [
    "constitution_id",
    "part_num",
    "section_num",
    "unit_num",
    "para_num",
    "sent_num",
    "token_num",
]

bags = dict(
    SENTS=OHCO[:6],
    PARAS=OHCO[:5],
    UNITS=OHCO[:4],
    SECTIONS=OHCO[:3],
    PARTS=OHCO[:2],
    CONSTITUTIONS=OHCO[:1],
)

In [4]:
bag = "CONSTITUTIONS"
# bag = "PARTS"
# bag = "SECTIONS"
# bag = "UNITS"

# Prepare the Data

## Import Tables

This matches the M05 pattern of reading the parsed M04-style output tables. The parsed notebook's save lines need to be uncommented before running this cell.

In [5]:
LIB = pd.read_csv(f"{source_dir}/{data_prefix}-LIB.csv", sep=CSV_DELIM).set_index("constitution_id")
TOKEN = pd.read_csv(f"{source_dir}/{data_prefix}-CORPUS.csv", sep=CSV_DELIM).set_index(OHCO)
VOCAB = pd.read_csv(f"{source_dir}/{data_prefix}-VOCAB.csv", sep=CSV_DELIM).set_index("term_str")

/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_45410/1113395148.py:2: DtypeWarning: Columns (0: section_type, 1: section_label, 2: unit_label, 3: unit_heading) have mixed types. Specify dtype option on import or set low_memory=False.
  TOKEN = pd.read_csv(f"{source_dir}/{data_prefix}-CORPUS.csv", sep=CSV_DELIM).set_index(OHCO)


In [6]:
TOKEN = TOKEN.dropna(subset=["term_str"]).copy()
TOKEN = TOKEN.loc[TOKEN.term_str.astype(str).str.len() > 0]
TOKEN.head()

country  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                
Afghanistan_2004 0        0           1        1        1        1          Afghanistan   
                                                                 2          Afghanistan   
                                                                 3          Afghanistan   
                                                                 4          Afghanistan   
                                                                 5          Afghanistan   

                                                                            file_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num              
Afghanistan_2004 0        0           1        1        1        1               2004   
                                                                 2               2004   
                                                                 3               2004   
                                                                 4               2004   
                                                                 5               2004   

                                                                            original_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                  
Afghanistan_2004 0        0           1        1        1        1                 2004.0   
                                                                 2                 2004.0   
                                                                 3                 2004.0   
                                                                 4                 2004.0   
                                                                 5                 2004.0   

                                                                            revision_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                  
Afghanistan_2004 0        0           1        1        1        1                    NaN   
                                                                 2                    NaN   
                                                                 3                    NaN   
                                                                 4                    NaN   
                                                                 5                    NaN   

                                                                                  title  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                
Afghanistan_2004 0        0           1        1        1        1          Afghanistan   
                                                                 2          Afghanistan   
                                                                 3          Afghanistan   
                                                                 4          Afghanistan   
                                                                 5          Afghanistan   

                                                                                       source_file_path  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                                
Afghanistan_2004 0        0           1        1        1        1          ./data/Afghanistan_2004.txt   
                                                                 2          ./data/Afghanistan_2004.txt   
                                                                 3          ./data/Afghanistan_2004.txt   
                                                                 4          ./data/Afghanistan_2004.txt   
                                                                 5          ./data/Afghanistan_2004.txt   

                                                                           part_type  \
constitution_id  part_num section_num 

In [7]:
VOCAB = VOCAB.sort_values("n", ascending=False).copy()
VOCAB["term_rank"] = np.arange(1, VOCAB.shape[0] + 1)
VOCAB["p"] = VOCAB.n / VOCAB.n.sum()
VOCAB["i"] = -np.log2(VOCAB.p)
VOCAB.head()

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem,df,idf,dfidf,dp,di,dh,term_rank
term_str,,,,,,,,,,,,,,,,,,,
the,429823,3,0.103120,3.277603,DT,DT,2,"{'NN', 'DT'}",2,"{'NNP', 'DT'}",1,the,192,0.0,0.0,1.0,0.0,0.0,1
of,296485,2,0.071131,3.813387,IN,IN,5,"{'NN', 'VB', 'CD', 'JJ', 'IN'}",6,"{'NN', 'CD', 'NNP', 'VBP', 'JJ', 'IN'}",1,of,192,0.0,0.0,1.0,0.0,0.0,2
and,129427,3,0.031051,5.009207,CC,CC,3,"{'NN', 'IN', 'CC'}",3,"{'NNP', 'IN', 'CC'}",1,and,192,0.0,0.0,1.0,0.0,0.0,3
to,117893,2,0.028284,5.143868,TO,TO,3,"{'NN', 'VB', 'TO'}",3,"{'NNP', 'VBP', 'TO'}",1,to,192,0.0,0.0,1.0,0.0,0.0,4
in,89792,2,0.021542,5.536687,IN,IN,2,"{'NN', 'IN'}",2,"{'NNP', 'IN'}",1,in,192,0.0,0.0,1.0,0.0,0.0,5


# Create BOW from TOKEN

Following M05, create `BOW` with a split-apply-combine pattern by grouping on the selected bag plus `term_str`.

In [8]:
bags[bag] + ["term_str"]

['constitution_id', 'term_str']

In [9]:
BOW = TOKEN.groupby(bags[bag] + ["term_str"]).term_str.count().to_frame("n")
BOW.head()

n
constitution_id  term_str    
Afghanistan_2004 1         20
                 10         3
                 11         1
                 111        1
                 112        1

# Document-Term Count Matrix

M05 calls this `DTCM`. The final project calls it `DTM`, so this notebook keeps both names: `DTCM` for class consistency and `DTM` for the project table.

In [10]:
DTCM = BOW.n.unstack(fill_value=0)
DTM = DTCM.astype(pd.SparseDtype("int", 0))
DTCM.head(10)

term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Albania_2008,0,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Algeria_2008,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Andorra_1993,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Angola_2010,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Antigua_and_Barbuda_1981,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Argentina_1994,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Armenia_2005,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Australia_1985,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Types and Tokens by DOC

In [11]:
DOC = DTCM.sum(1).to_frame("n_tokens")
DOC["n_types"] = DTCM.astype("bool").sum(1)
DOC["pkr"] = DOC.n_types / DOC.n_tokens

if bag == "CONSTITUTIONS":
    DOC = DOC.join(LIB[["country", "file_year", "original_year", "revision_year", "title"]])

DOC.sort_values("pkr").head(20)

/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_45410/1205689900.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  DOC = DTCM.sum(1).to_frame("n_tokens")
/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_45410/1205689900.py:2: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  DOC["n_types"] = DTCM.astype("bool").sum(1)


,n_tokens,n_types,pkr,country,file_year,original_year,revision_year,title
constitution_id,,,,,,,,
India_2012,101401,3725,0.036735,India,2012,1949.0,2012.0,India
St_Kitts_and_Nevis_1983,45856,2072,0.045185,St Kitts and Nevis,1983,1983.0,NaN,St. Kitts and Nevis
Malaysia_1996,62859,3049,0.048505,Malaysia,1996,1957.0,1996.0,Malaysia
St_Lucia_1978,39683,1950,0.049139,St Lucia,1978,1978.0,NaN,St. Lucia
Fiji_2013,46381,2337,0.050387,Fiji,2013,2013.0,NaN,Fiji
Jamaica_1994,37714,1939,0.051413,Jamaica,1994,1962.0,1994.0,Jamaica
Singapore_2010,43355,2231,0.051459,Singapore,2010,1959.0,2010.0,Singapore
Lesotho_1998,42027,2184,0.051967,Lesotho,1998,1993.0,1998.0,Lesotho
Sweden_2012,57604,3042,0.052809,Sweden,2012,1974.0,2012.0,Sweden


# Compute TFIDF

In [12]:
tf_method = "double_norm"       # sum, max, log, double_norm, raw, binary
tf_norm_k = .5                  # only used for double_norm
idf_method = "standard"        # standard, max, plus, smooth

## Compute TF

In [13]:
print("TF method:", tf_method)

if tf_method == "sum":
    TF = DTCM.T / DTCM.T.sum()

elif tf_method == "smooth":
    TF = (DTCM.T / DTCM.T.sum()) + 1

elif tf_method == "max":
    TF = DTCM.T / DTCM.T.max()

elif tf_method == "log":
    TF = np.log2(1 + DTCM.T)

elif tf_method == "raw":
    TF = DTCM.T

elif tf_method == "double_norm":
    TF = ((DTCM.T + tf_norm_k) / (DTCM.T.max() + tf_norm_k)) + tf_norm_k

elif tf_method == "binary":
    TF = DTCM.T.astype("bool").astype("int")

TF = TF.T
TF.head()

TF method: double_norm


term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,...,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440,0.500440
Albania_2008,0.500323,0.500323,0.501614,0.500323,0.500323,0.500323,0.500323,0.500323,0.500323,0.500323,...,0.500323,0.500323,0.500323,0.500323,0.500323,0.500323,0.500323,0.500323,0.500323,0.500323
Algeria_2008,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,...,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320,0.500320
Andorra_1993,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,...,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450,0.500450
Angola_2010,0.500175,0.500175,0.500526,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,...,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175


## Compute DF

In [14]:
DF = DTCM.astype("bool").sum()
DF.sort_values(ascending=False).head(20)

term_str
of            192
duties        192
government    192
no            192
other         192
with          192
take          192
a             192
general       192
order         192
or            192
within        192
right         192
its           192
one           192
rights        192
on            192
all           192
from          192
state         192
dtype: int64

## Compute IDF

In [15]:
N = DTCM.shape[0]

In [16]:
print("IDF method:", idf_method)

if idf_method == "standard":
    IDF = np.log2(N / DF)

elif idf_method == "max":
    IDF = np.log2(DF.max() / DF)

elif idf_method == "plus":
    IDF = np.log2(N / DF) + 1

elif idf_method == "smooth":
    IDF = np.log2((1 + N) / (1 + DF)) + 1

IDF.sort_values(ascending=False).head(20)

IDF method: standard


term_str
zuru           7.584963
marwat         7.584963
subisse        7.584963
subhuman       7.584963
maryborough    7.584963
enderbury      7.584963
subdued        7.584963
mary           7.584963
subdelegate    7.584963
endorses       7.584963
marx           7.584963
maru           7.584963
subitem        7.584963
endower        7.584963
replaceable    7.584963
subclauses     7.584963
endownments    7.584963
endurance      7.584963
martin         7.584963
martian        7.584963
dtype: float64

## Compute TFIDF

In [17]:
TFIDF = TF * IDF
TFIDF.head()

term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,2.294497,2.794936,1.016079,3.795815,3.295376,3.295376,3.795815,3.795815,3.795815,2.209459,...,3.795815,3.795815,3.795815,3.795815,3.795815,3.795815,3.295376,3.295376,3.795815,3.795815
Albania_2008,2.293962,2.794285,1.018465,3.794930,3.294607,3.294607,3.794930,3.794930,3.794930,2.208944,...,3.794930,3.794930,3.794930,3.794930,3.794930,3.794930,3.294607,3.294607,3.794930,3.794930
Algeria_2008,2.293949,2.794270,1.015837,3.794910,3.294590,3.294590,3.794910,3.794910,3.794910,2.208932,...,3.794910,3.794910,3.794910,3.794910,3.794910,3.794910,3.294590,3.294590,3.794910,3.794910
Andorra_1993,2.294546,2.794996,1.016101,3.795896,3.295446,3.295446,3.795896,3.795896,3.795896,2.209507,...,3.795896,3.795896,3.795896,3.795896,3.795896,3.795896,3.295446,3.295446,3.795896,3.795896
Angola_2010,2.293286,2.793461,1.016256,3.793812,3.293637,3.293637,3.793812,3.793812,3.793812,2.208293,...,3.793812,3.793812,3.793812,3.793812,3.793812,3.793812,3.293637,3.293637,3.793812,3.793812


## Move Things to Their Places

In [18]:
VOCAB["df"] = DF
VOCAB["idf"] = IDF
VOCAB["tfidf_mean"] = TFIDF.mean()

BOW["tf"] = TF.stack().reindex(BOW.index)
BOW["tfidf"] = TFIDF.stack().reindex(BOW.index)

BOW.head()

n        tf     tfidf
constitution_id  term_str                        
Afghanistan_2004 1         20  0.518022  0.023727
                 10         3  0.503077  0.141456
                 11         1  0.501319  0.282039
                 111        1  0.501319  1.049315
                 112        1  0.501319  1.065568

# Apply

In [19]:
TFIDF_BY_CONSTITUTION = TFIDF.groupby("constitution_id").mean()

def top_constitutions_for_term(term_str):
    return TFIDF_BY_CONSTITUTION[term_str]\
        .to_frame("tfidf_mean")\
        .join(LIB)\
        .sort_values("tfidf_mean", ascending=False)\
        [["country", "file_year", "title", "tfidf_mean"]]

top_constitutions_for_term("rights").head(20)

,country,file_year,title,tfidf_mean
constitution_id,,,,
Afghanistan_2004,Afghanistan,2004,Afghanistan,0.0
Albania_2008,Albania,2008,Albania,0.0
Nepal_2010,Nepal,2010,Nepal,0.0
Netherlands_2008,Netherlands,2008,Netherlands,0.0
Nicaragua_2005,Nicaragua,2005,Nicaragua,0.0
Niger_2010,Niger,2010,Niger,0.0
Nigeria_1999,Nigeria,1999,Nigeria,0.0
Norway_2004,Norway,2004,Norway,0.0
Oman_2011,Oman,2011,Oman,0.0


# Apply Aggregates to VOCAB

This follows the M05 aggregate TFIDF notebook. `dfidf` depends only on document frequency, so ties are expected.

In [20]:
VOCAB["dfidf"] = VOCAB.df * VOCAB.idf
VOCAB["dp"] = VOCAB.df / N
VOCAB["di"] = np.log2(1 / VOCAB.dp)
VOCAB["dh"] = VOCAB.dp * VOCAB.di

VOCAB.sort_values(["dfidf", "n"], ascending=False).head(20)

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem,df,idf,dfidf,dp,di,dh,term_rank,tfidf_mean
term_str,,,,,,,,,,,,,,,,,,,,
assent,622,6,0.000149,12.710216,NN,NN,3,"{'NN', 'VB', 'JJ'}",5,"{'NN', 'VB', 'NNP', 'JJ', 'VBD'}",0,assent,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,730,0.719802
labor,496,5,0.000119,13.036790,NN,NN,1,{'NN'},2,"{'NN', 'NNP'}",0,labor,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,877,0.719654
rural,397,5,0.000095,13.357991,JJ,JJ,3,"{'NN', 'VB', 'JJ'}",3,"{'VB', 'NNP', 'JJ'}",0,rural,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1035,0.718991
hand,348,4,0.000083,13.548043,NN,NN,1,{'NN'},2,"{'NN', 'NNP'}",0,hand,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1157,0.719105
like,303,4,0.000073,13.747813,JJ,JJ,3,"{'VB', 'IN', 'JJ'}",3,"{'VB', 'IN', 'JJ'}",0,like,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1282,0.718889
organized,299,9,0.000072,13.766985,VBN,VB,3,"{'VB', 'NN', 'JJ'}",4,"{'NNP', 'VBD', 'VBN', 'JJ'}",0,organ,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1291,0.719070
class,291,5,0.000070,13.806111,NN,NN,1,{'NN'},2,"{'NN', 'NNP'}",0,class,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1320,0.718954
fees,251,4,0.000060,14.019443,NNS,NN,1,{'NN'},1,{'NNS'},0,fee,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1480,0.718757
magistrate,233,10,0.000056,14.126800,NN,NN,3,"{'NN', 'VB', 'JJ'}",5,"{'NN', 'VB', 'NNP', 'VBP', 'JJ'}",0,magistr,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1544,0.718765


# Reduce VOCAB

Following M05, use document entropy `dh` to define a threshold for significant terms.

In [21]:
thresh = VOCAB.dh.quantile(.9).round(3)
float(thresh)

0.437

In [22]:
SIGS = VOCAB[VOCAB.dh >= thresh].sort_values("dh", ascending=False)
SIGS.shape

(2416, 20)

In [23]:
SIGS[["max_pos", "max_pos_group", "n", "i", "df", "idf", "dfidf", "dh"]].head(25)

,max_pos,max_pos_group,n,i,df,idf,dfidf,dh
term_str,,,,,,,,
delivered,VBN,VB,221,14.203084,71.0,1.435215,101.900292,0.530731
integral,JJ,JJ,179,14.507171,71.0,1.435215,101.900292,0.530731
replacement,NN,NN,141,14.851435,71.0,1.435215,101.900292,0.530731
thereby,RB,RB,124,15.036790,71.0,1.435215,101.900292,0.530731
hand,NN,NN,348,13.548043,71.0,1.435215,101.900292,0.530731
applying,VBG,VB,142,14.841239,71.0,1.435215,101.900292,0.530731
read,VBN,VB,202,14.332775,71.0,1.435215,101.900292,0.530731
regardless,NN,NN,144,14.821061,71.0,1.435215,101.900292,0.530731
messages,NNS,NN,107,15.249520,71.0,1.435215,101.900292,0.530731


# Reduced and Normalized TFIDF_L2

In [24]:
TFIDF_REDUCED = TFIDF[SIGS.index]
TFIDF_REDUCED.head()

term_str,delivered,integral,replacement,thereby,hand,applying,read,regardless,messages,stage,...,maximum,includes,punished,equally,enforcement,union,functioning,suffrage,restrictions,organizations
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.718239,0.718239,0.718239,0.718239,0.718239,0.718239,0.718239,0.718239,0.718239,0.718239,...,0.364449,0.364449,0.365728,0.363810,0.365089,0.363810,0.363810,0.363810,0.364449,0.368284
Albania_2008,0.718071,0.718071,0.718071,0.718071,0.718071,0.718071,0.718071,0.718071,0.718998,0.718071,...,0.363725,0.364195,0.364195,0.364195,0.363725,0.363725,0.363725,0.363725,0.363725,0.369829
Algeria_2008,0.718986,0.718986,0.720825,0.718067,0.718067,0.718986,0.718067,0.718067,0.718067,0.718067,...,0.366517,0.363724,0.365120,0.363724,0.363724,0.364189,0.368379,0.365120,0.363724,0.363724
Andorra_1993,0.718254,0.718254,0.718254,0.718254,0.718254,0.719546,0.718254,0.719546,0.718254,0.718254,...,0.365782,0.363818,0.363818,0.363818,0.366437,0.364473,0.370365,0.365127,0.363818,0.365127
Angola_2010,0.718363,0.718363,0.718867,0.717860,0.718363,0.718363,0.717860,0.718363,0.717860,0.717860,...,0.363873,0.363873,0.363873,0.363873,0.363618,0.365659,0.373058,0.365149,0.365659,0.364384


In [25]:
row_norms = np.sqrt((TFIDF_REDUCED ** 2).sum(1))
TFIDF_L2 = TFIDF_REDUCED.div(row_norms.replace(0, np.nan), axis=0).fillna(0)
TFIDF_L2.head()

/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_45410/3970364029.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  row_norms = np.sqrt((TFIDF_REDUCED ** 2).sum(1))


term_str,delivered,integral,replacement,thereby,hand,applying,read,regardless,messages,stage,...,maximum,includes,punished,equally,enforcement,union,functioning,suffrage,restrictions,organizations
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,...,0.008493,0.008493,0.008523,0.008478,0.008508,0.008478,0.008478,0.008478,0.008493,0.008583
Albania_2008,0.016739,0.016739,0.016739,0.016739,0.016739,0.016739,0.016739,0.016739,0.016760,0.016739,...,0.008479,0.008490,0.008490,0.008490,0.008479,0.008479,0.008479,0.008479,0.008479,0.008621
Algeria_2008,0.016761,0.016761,0.016804,0.016740,0.016740,0.016761,0.016740,0.016740,0.016740,0.016740,...,0.008544,0.008479,0.008512,0.008479,0.008479,0.008490,0.008588,0.008512,0.008479,0.008479
Andorra_1993,0.016739,0.016739,0.016739,0.016739,0.016739,0.016769,0.016739,0.016769,0.016739,0.016739,...,0.008524,0.008479,0.008479,0.008479,0.008540,0.008494,0.008631,0.008509,0.008479,0.008509
Angola_2010,0.016747,0.016747,0.016759,0.016735,0.016747,0.016747,0.016735,0.016747,0.016735,0.016735,...,0.008483,0.008483,0.008483,0.008483,0.008477,0.008524,0.008697,0.008513,0.008524,0.008495


In [26]:
np.sqrt((TFIDF_L2 ** 2).sum(1)).describe()

/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_45410/911463411.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  np.sqrt((TFIDF_L2 ** 2).sum(1)).describe()


count    1.920000e+02
mean     1.000000e+00
std      4.055786e-15
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

# Inspect Tables

In [27]:
BOW.head(20)

n        tf     tfidf
constitution_id  term_str                        
Afghanistan_2004 1         20  0.518022  0.023727
                 10         3  0.503077  0.141456
                 11         1  0.501319  0.282039
                 111        1  0.501319  1.049315
                 112        1  0.501319  1.065568
                 118        1  0.501319  1.252043
                 12         1  0.501319  0.233625
                 127        2  0.502198  1.298162
                 13         1  0.501319  0.322071
                 134        1  0.501319  1.295890
                 1381       1  0.501319  3.802483
                 14         1  0.501319  0.276497
                 144        1  0.501319  1.474432
                 145        1  0.501319  1.600533
                 146        1  0.501319  1.712022
                 15         4  0.503956  0.169851
                 159        1  0.501319  1.566887
                 16         1  0.501319  0.339930
                 17         1  0.501319  0.333928
                 18         1  0.501319  0.228441

In [28]:
DTM.head(10)

term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Albania_2008,0,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Algeria_2008,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Andorra_1993,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Angola_2010,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Antigua_and_Barbuda_1981,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Argentina_1994,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Armenia_2005,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Australia_1985,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
TFIDF.head(10)

term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,2.294497,2.794936,1.016079,3.795815,3.295376,3.295376,3.795815,3.795815,3.795815,2.209459,...,3.795815,3.795815,3.795815,3.795815,3.795815,3.795815,3.295376,3.295376,3.795815,3.795815
Albania_2008,2.293962,2.794285,1.018465,3.794930,3.294607,3.294607,3.794930,3.794930,3.794930,2.208944,...,3.794930,3.794930,3.794930,3.794930,3.794930,3.794930,3.294607,3.294607,3.794930,3.794930
Algeria_2008,2.293949,2.794270,1.015837,3.794910,3.294590,3.294590,3.794910,3.794910,3.794910,2.208932,...,3.794910,3.794910,3.794910,3.794910,3.794910,3.794910,3.294590,3.294590,3.794910,3.794910
Andorra_1993,2.294546,2.794996,1.016101,3.795896,3.295446,3.295446,3.795896,3.795896,3.795896,2.209507,...,3.795896,3.795896,3.795896,3.795896,3.795896,3.795896,3.295446,3.295446,3.795896,3.795896
Angola_2010,2.293286,2.793461,1.016256,3.793812,3.293637,3.293637,3.793812,3.793812,3.793812,2.208293,...,3.793812,3.793812,3.793812,3.793812,3.793812,3.793812,3.293637,3.293637,3.793812,3.793812
Antigua_and_Barbuda_1981,2.293247,2.793414,1.015526,3.793749,3.293581,3.293581,3.793749,3.793749,3.793749,2.208256,...,3.793749,3.793749,3.793749,3.793749,3.793749,3.793749,3.293581,3.293581,3.793749,3.793749
Argentina_1994,2.294141,2.794503,1.017391,3.795226,3.294865,3.294865,3.795226,3.795226,3.795226,2.209117,...,3.795226,3.795226,3.795226,3.795226,3.795226,3.795226,3.294865,3.294865,3.795226,3.795226
Armenia_2005,2.293722,2.793993,1.015736,3.794534,3.294263,3.294263,3.794534,3.794534,3.794534,2.208714,...,3.794534,3.794534,3.794534,3.794534,3.794534,3.794534,3.294263,3.294263,3.794534,3.794534
Australia_1985,2.293981,2.794308,1.015851,3.794962,3.294635,3.294635,3.794962,3.794962,3.794962,2.208963,...,3.794962,3.794962,3.794962,3.794962,3.794962,3.794962,3.294635,3.294635,3.794962,3.794962


In [30]:
TFIDF_L2.head(10)

term_str,delivered,integral,replacement,thereby,hand,applying,read,regardless,messages,stage,...,maximum,includes,punished,equally,enforcement,union,functioning,suffrage,restrictions,organizations
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,0.016738,...,0.008493,0.008493,0.008523,0.008478,0.008508,0.008478,0.008478,0.008478,0.008493,0.008583
Albania_2008,0.016739,0.016739,0.016739,0.016739,0.016739,0.016739,0.016739,0.016739,0.016760,0.016739,...,0.008479,0.008490,0.008490,0.008490,0.008479,0.008479,0.008479,0.008479,0.008479,0.008621
Algeria_2008,0.016761,0.016761,0.016804,0.016740,0.016740,0.016761,0.016740,0.016740,0.016740,0.016740,...,0.008544,0.008479,0.008512,0.008479,0.008479,0.008490,0.008588,0.008512,0.008479,0.008479
Andorra_1993,0.016739,0.016739,0.016739,0.016739,0.016739,0.016769,0.016739,0.016769,0.016739,0.016739,...,0.008524,0.008479,0.008479,0.008479,0.008540,0.008494,0.008631,0.008509,0.008479,0.008509
Angola_2010,0.016747,0.016747,0.016759,0.016735,0.016747,0.016747,0.016735,0.016747,0.016735,0.016735,...,0.008483,0.008483,0.008483,0.008483,0.008477,0.008524,0.008697,0.008513,0.008524,0.008495
Antigua_and_Barbuda_1981,0.016742,0.016731,0.016742,0.016742,0.016764,0.016753,0.016775,0.016742,0.016731,0.016753,...,0.008480,0.008565,0.008480,0.008486,0.008492,0.008475,0.008475,0.008475,0.008554,0.008475
Argentina_1994,0.016732,0.016781,0.016732,0.016756,0.016732,0.016756,0.016732,0.016732,0.016756,0.016732,...,0.008475,0.008475,0.008500,0.008475,0.008475,0.008512,0.008475,0.008488,0.008475,0.008512
Armenia_2005,0.016740,0.016740,0.016740,0.016740,0.016740,0.016740,0.016740,0.016758,0.016740,0.016740,...,0.008497,0.008488,0.008479,0.008479,0.008479,0.008479,0.008497,0.008497,0.008516,0.008516
Australia_1985,0.016757,0.016735,0.016735,0.016735,0.016757,0.016735,0.016757,0.016735,0.016735,0.016757,...,0.008521,0.008488,0.008477,0.008499,0.008477,0.008521,0.008477,0.008488,0.008488,0.008477


# Save

The export lines are intentionally commented out so the derived tables can be inspected before saving.

In [31]:
save_path = f"{output_dir}/{data_prefix}"

# BOW.to_csv(f"{save_path}-BOW-{bag}.csv", sep=CSV_DELIM)
# DTM.to_csv(f"{save_path}-DTM-{bag}.csv", sep=CSV_DELIM)
# DOC.to_csv(f"{save_path}-DOC-{bag}.csv", sep=CSV_DELIM)
# TFIDF.to_csv(f"{save_path}-TFIDF-{bag}.csv", sep=CSV_DELIM)
# VOCAB.to_csv(f"{save_path}-VOCAB-{bag}.csv", sep=CSV_DELIM)
# SIGS.to_csv(f"{save_path}-SIGS-{bag}.csv", sep=CSV_DELIM)
# TFIDF_REDUCED.to_csv(f"{save_path}-TFIDF_REDUCED-{bag}.csv", sep=CSV_DELIM)
# TFIDF_L2.to_csv(f"{save_path}-TFIDF_L2-{bag}.csv", sep=CSV_DELIM)